# CE541E08 — Unit 3 · Day 26 — Reading and Writing Array Data on Files

| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 3 — The NumPy Library |
| **Session** | Day 26 of 45 |
| **CO** | CO3, CO4 |
| **Topics** | np.savetxt · np.loadtxt · np.genfromtxt · Flow Duration Curve |

---
> Read the explanation before each code block. Check the expected output. Run the cell and verify. Then try the small challenge at the end.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
session      = "Day 26"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — File I/O with NumPy

In engineering practice, data comes from files — CSV exports from data loggers, CWC gauge records, IMD rainfall archives. NumPy provides two functions for reading and writing:

| Function | Use case |
|---|---|
| `np.savetxt(file, data, ...)` | Write a NumPy array to a text/CSV file |
| `np.loadtxt(file, ...)` | Read a clean CSV (no missing values) |
| `np.genfromtxt(file, ...)` | Read a CSV that may have missing values |

Today we save a streamflow dataset to CSV, reload it, handle missing values, and build the Flow Duration Curve — a standard hydrological analysis tool.

---
## Code Block 1 — Saving Data to CSV with np.savetxt

### What this code does

We build a 31-day streamflow table with two columns (Day and Flow) using `np.column_stack` and save it to a CSV file using `np.savetxt`. This is the standard way to export NumPy data for sharing with other engineers or loading into Excel.

### Why each step is taken

**`np.column_stack([days, flow])`:**
`column_stack` places two 1-D arrays side by side as columns, producing a `(31,2)` matrix. This is the correct way to assemble a two-column table from separate arrays. Using `np.vstack` would stack them as rows instead.

**`np.savetxt(filename, data, delimiter, fmt, header, comments)`:**
- `delimiter=','` — produces CSV (comma-separated)
- `fmt='%.1f'` — one decimal place for all values
- `header='Day,Flow_m3s'` — adds a header row
- `comments=''` — by default NumPy prefixes the header with `#`; setting `comments=''` removes that prefix so Excel can read the header correctly

**Printing `data_out[:5]`:**
Verifying the first few rows before saving is good practice — it confirms the array is structured as expected.

### Algorithm

```
1. Create day numbers: np.arange(1,32) → [1,2,...,31]

2. Define 31-day flow array

3. np.column_stack([days, flow])
   → (31,2) matrix: col 0 = day, col 1 = flow

4. np.savetxt('/tmp/cwc_july.csv', data_out,
              delimiter=',', fmt='%.1f',
              header='Day,Flow_m3s', comments='')
   → writes CSV file with header

5. Print confirmation and first 5 rows
```

### Expected output

```
Saved cwc_july.csv
First 5 rows:
[[   1.  234.5]
 [   2.  267.8]
 [   3.  312.4]
 [   4.  890.2]
 [   5. 1245.6]]
```

In [ ]:
import numpy as np

# 31-day streamflow record (m³/s) — KRS station, July
days = np.arange(1, 32)
flow = np.array([234.5, 267.8, 312.4, 890.2, 1245.6, 987.3, 756.4,
                 543.2, 412.8, 345.6, 289.4, 245.1, 212.3, 198.7,
                 212.3, 178.9, 156.4, 143.2, 189.6, 234.5, 267.8,
                 312.4, 456.7, 678.9, 890.2, 1123.4, 987.3, 765.4,
                 543.2, 412.8, 345.6])

# np.column_stack places arrays side-by-side as columns → (31,2) matrix
data_out = np.column_stack([days, flow])

# np.savetxt writes the array to a text file
# comments='' prevents NumPy from adding '#' before the header row
np.savetxt('/tmp/cwc_july.csv', data_out,
           delimiter=',', fmt='%.1f',
           header='Day,Flow_m3s', comments='')

print("Saved cwc_july.csv")
print("First 5 rows:"); print(data_out[:5])

### 🔁 Try this

Open the saved file to verify its contents:

```python
with open('/tmp/cwc_july.csv') as f:
    print(f.read())
```

Check that the header row does not have a `#` prefix and that the values match.

---
## Code Block 2 — Loading Clean Data with np.loadtxt

### What this code does

We reload the CSV file we just saved and perform a basic analysis — extracting the flow column, computing statistics, and identifying which day had the peak discharge.

### Why each step is taken

**`np.loadtxt(file, delimiter, skiprows)`:**
`skiprows=1` skips the header row. The result is a `(31,2)` float array. We then slice columns using `data[:,0]` for Day and `data[:,1]` for Flow.

**`data[:,0].astype(int)`:**
The day column was saved and loaded as `float` (1.0, 2.0, ...). Converting to `int` gives clean day numbers for printing.

**`days[flow.argmax()]`:**
`flow.argmax()` gives the index of the maximum flow value. Using that index into the `days` array gives the actual day number — more reliable than adding 1 to the index, which only works if the data starts at Day 1.

### Algorithm

```
1. np.loadtxt('/tmp/cwc_july.csv', delimiter=',', skiprows=1)
   → (31,2) float array

2. days = data[:,0].astype(int)   → column 0 = day numbers
   flow = data[:,1]               → column 1 = flow values

3. flow.mean() → mean discharge
   flow.max()  → peak discharge
   days[flow.argmax()] → day number of peak

4. Print summary statistics
```

### Expected output

```
Loaded shape: (31, 2)
Mean Q : 517.5 m3/s
Max Q  : 1245.6 m3/s on Day 5
```

In [ ]:
import numpy as np

# np.loadtxt reads a clean CSV into a NumPy array
# skiprows=1: skip the header line ('Day,Flow_m3s')
data = np.loadtxt('/tmp/cwc_july.csv', delimiter=',', skiprows=1)

print(f"Loaded shape: {data.shape}")

# Extract columns by index
days = data[:, 0].astype(int)   # column 0 = day numbers (convert float→int)
flow = data[:, 1]               # column 1 = flow values

print(f"Mean Q : {flow.mean():.1f} m3/s")
print(f"Max Q  : {flow.max():.1f} m3/s on Day {days[flow.argmax()]}")

### 🔁 Try this

Compute and print the number of days where flow exceeds the mean.

Use `(flow > flow.mean()).sum()` — then express this as a percentage of 31 days.

---
## Code Block 3 — Handling Missing Values with np.genfromtxt

### What this code does

Real datasets have missing values — sensor outages, transmission errors, manual gaps. We create a CSV with `-999` as the missing-value flag, load it with `np.genfromtxt`, replace the flags with `np.nan`, and compute statistics on the valid data.

### Why each step is taken

**`np.genfromtxt` vs `np.loadtxt`:**
`loadtxt` fails if any value cannot be converted to a number. `genfromtxt` is more flexible — it can handle missing values, mixed formats, and inconsistent column counts. `filling_values=-999` replaces any missing/blank cell with -999 when loading.

**`np.where(flow == -999, np.nan, flow)`:**
After loading, we replace the -999 sentinel with `np.nan` using `np.where`. This is cleaner than a loop. Now all nan-aware NumPy functions can work correctly.

**`np.nanmean` and `np.isnan`:**
`np.nanmean` skips NaN values when computing the mean. `np.isnan(flow_c).sum()` counts how many values are NaN — i.e. how many records are missing.

### Algorithm

```
1. Write a small CSV string with -999 missing values to a file

2. np.genfromtxt(file, delimiter, skip_header, filling_values=-999)
   → loads data; blanks replaced with -999

3. flow = data[:,1]

4. np.where(flow == -999, np.nan, flow)
   → replace -999 sentinel with np.nan

5. np.nanmean(flow_c) → mean of valid values
   np.isnan(flow_c).sum() → count of missing records
```

### Expected output

```
Valid mean: 537.47 m3/s
Missing   : 2
```

In [ ]:
import numpy as np

# Create a small CSV with -999 as the missing-value flag
raw = "Day,Flow\n1,234.5\n2,267.8\n3,-999\n4,890.2\n5,-999\n6,756.4"
with open('/tmp/cwc_miss.csv', 'w') as f:
    f.write(raw)

# np.genfromtxt handles imperfect CSVs
# filling_values=-999: any blank/unreadable cell becomes -999
data  = np.genfromtxt('/tmp/cwc_miss.csv', delimiter=',',
                      skip_header=1, filling_values=-999)
flow  = data[:, 1]

# Replace -999 sentinel with np.nan — float-safe missing value marker
# np.where(condition, value_if_true, value_if_false)
flow_c = np.where(flow == -999, np.nan, flow)

print(f"Valid mean: {np.nanmean(flow_c):.2f} m3/s")
print(f"Missing   : {np.isnan(flow_c).sum()}")

### 🔁 Try this

After replacing missing values with NaN, fill the gaps with the valid data mean:

```python
flow_filled = np.where(np.isnan(flow_c), np.nanmean(flow_c), flow_c)
```

Print the filled array and verify the two missing positions now have the mean value.

---
## Code Block 4 — Flow Duration Curve (FDC)

### What this code does

We build the Flow Duration Curve (FDC) from the 31-day streamflow record. The FDC plots discharge (y-axis) against exceedance probability (x-axis) — showing what flow is exceeded what percentage of the time. We compute Q10, Q50, and Q90 and calculate the flashiness index.

### Why each step is taken

**Sorting descending `np.sort(flow)[::-1]`:**
The FDC is constructed by ranking flows from highest to lowest. `np.sort` sorts ascending; `[::-1]` reverses to descending. The highest flow gets rank 1 (lowest exceedance probability).

**Weibull plotting position `P = rank/(n+1)*100`:**
For each ranked flow, its exceedance probability is `rank/(n+1)`. Multiplying by 100 gives a percentage. This is the standard Weibull formula — it avoids P=0 and P=100 which would plot at ±infinity on a log scale.

**`np.searchsorted(P, 10)`:**
Finds the index where the value 10 would be inserted to keep `P` sorted. Since P is already sorted ascending, this gives the index of the first exceedance probability ≥ 10% — corresponding to Q10 (the flow exceeded 10% of the time).

**Flashiness index Q10/Q90:**
A high Q10/Q90 ratio means the river has very high flows some of the time and very low flows most of the time — it is "flashy". A ratio close to 1 means steady, perennial flow.

**`plt.semilogy`:**
Plots y-axis on a logarithmic scale — standard for FDC because flows span several orders of magnitude. Linear scale would compress the low-flow part of the curve.

### Algorithm

```
1. Sort flow descending: flow_s = np.sort(flow)[::-1]
   n = 31

2. Weibull exceedance probability:
   rank = np.arange(1, n+1)
   P    = rank / (n+1) * 100   → percentages from ~3% to ~97%

3. Q10 = flow_s[np.searchsorted(P, 10)]
   Q50 = flow_s[np.searchsorted(P, 50)]
   Q90 = flow_s[np.searchsorted(P, 90)]

4. Flashiness = Q10 / Q90

5. Plot on semi-log axes: P (x) vs flow_s (y)
   Add horizontal lines at Q10, Q50, Q90
```

### Expected output

```
Q10=987.3  Q50=312.4  Q90=189.6 m3/s
Flashiness Q10/Q90=5.21
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

flow = np.array([234.5, 267.8, 312.4, 890.2, 1245.6, 987.3, 756.4,
                 543.2, 412.8, 345.6, 289.4, 245.1, 212.3, 198.7,
                 212.3, 178.9, 156.4, 143.2, 189.6, 234.5, 267.8,
                 312.4, 456.7, 678.9, 890.2, 1123.4, 987.3, 765.4,
                 543.2, 412.8, 345.6])

n = len(flow)

# Sort descending: highest flow gets rank 1
flow_s = np.sort(flow)[::-1]

# Weibull plotting position: P = rank/(n+1) × 100 (%)
# Avoids P=0 and P=100 which are undefined on log scale
P = np.arange(1, n+1) / (n + 1) * 100

# np.searchsorted finds the insertion index in a sorted array
# Since P is sorted ascending, searchsorted(P, 10) gives index of first P ≥ 10%
Q10 = flow_s[np.searchsorted(P, 10)]
Q50 = flow_s[np.searchsorted(P, 50)]
Q90 = flow_s[np.searchsorted(P, 90)]

print(f"Q10={Q10:.1f}  Q50={Q50:.1f}  Q90={Q90:.1f} m3/s")
print(f"Flashiness Q10/Q90={Q10/Q90:.2f}")

# Semi-log FDC plot — log y-axis standard for FDC
plt.figure(figsize=(8, 5))
plt.semilogy(P, flow_s, 'b-o', markersize=3)
for Q, lbl, col in [(Q10,'Q10','orange'), (Q50,'Q50','green'), (Q90,'Q90','red')]:
    plt.axhline(Q, color=col, linestyle='--', label=f'{lbl}={Q:.0f}')
plt.xlabel('Exceedance Probability (%)')
plt.ylabel('Discharge (m3/s) — log scale')
plt.title('Flow Duration Curve — KRS Station, July')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 🔁 Try this

Compute Q25 (flow exceeded 25% of the time) and Q75 (exceeded 75% of the time).

- What is the ratio Q25/Q75?
- How does this compare to Q10/Q90?
- Add horizontal lines for Q25 and Q75 to the plot.

---
## Session Summary — File I/O and Flow Duration Curve

| Function | Syntax | Use case |
|---|---|---|
| Save to CSV | `np.savetxt(file, arr, delimiter=',', fmt, header, comments='')` | Export data |
| Load clean CSV | `np.loadtxt(file, delimiter=',', skiprows=1)` | Read data without gaps |
| Load with gaps | `np.genfromtxt(file, delimiter, skip_header, filling_values)` | Handle missing data |
| Stack as columns | `np.column_stack([a,b])` | Build multi-column table |
| Extract column | `data[:,1]` | Select one column from loaded array |
| Sort descending | `np.sort(arr)[::-1]` | Rank from highest to lowest |
| Search sorted | `np.searchsorted(P, value)` | Find index of a percentile |
| Semi-log plot | `plt.semilogy(x, y)` | FDC — y-axis on log scale |

---
## Day 26 Assignment

Use the 31-day July streamflow from Code Block 1.

1. Add a third column `Category` (0=normal, 1=high if flow>600, 2=flood if flow>1000) using `np.where`
2. Save the 3-column array to `/tmp/flow_cat.csv` with header `Day,Flow,Cat`
3. Reload the file and verify the shape is `(31,3)`
4. From the reloaded data, compute Q10, Q50, Q90 and the flashiness index

### ▶ Assignment cell

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

flow = np.array([234.5, 267.8, 312.4, 890.2, 1245.6, 987.3, 756.4,
                 543.2, 412.8, 345.6, 289.4, 245.1, 212.3, 198.7,
                 212.3, 178.9, 156.4, 143.2, 189.6, 234.5, 267.8,
                 312.4, 456.7, 678.9, 890.2, 1123.4, 987.3, 765.4,
                 543.2, 412.8, 345.6])
days = np.arange(1, 32)

# 1. Category column
cat = np.where(flow > 1000, 2, np.where(flow > 600, 1, 0))

# 2. Save 3-column array
np.savetxt('/tmp/flow_cat.csv',
           np.column_stack([days, flow, cat]),
           delimiter=',', fmt=['%d','%.1f','%d'],
           header='Day,Flow,Cat', comments='')

# 3. Reload and verify
data = np.loadtxt('/tmp/flow_cat.csv', delimiter=',', skiprows=1)
print(f"Reloaded shape: {data.shape}")

# 4. FDC from reloaded data
fl   = data[:, 1]
n    = len(fl)
fl_s = np.sort(fl)[::-1]
P    = np.arange(1, n+1) / (n+1) * 100
Q10  = fl_s[np.searchsorted(P, 10)]
Q50  = fl_s[np.searchsorted(P, 50)]
Q90  = fl_s[np.searchsorted(P, 90)]
print(f"Q10={Q10:.1f} Q50={Q50:.1f} Q90={Q90:.1f}")
print(f"Flashiness={Q10/Q90:.2f}")

---
- [ ] Run all cells from top to bottom — verify outputs match expected outputs above
- [ ] Complete the assignment cell
- [ ] Upload to GitHub: `Unit3_NumPy/CE541E08_U3_Day26.ipynb`
- [ ] Commit message: `Day 26 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*